# ДЗ4: Прогнозування міцності бетону за допомогою нейронних мереж PyTorch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import kagglehub

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Підготовка даних

Завантажуємо набір даних **Concrete Strength Prediction** з Kaggle. Датасет містить 8 числових ознак (компоненти бетонної суміші та вік) і цільову змінну — міцність бетону (МПа).

In [ ]:
path = kagglehub.dataset_download("mchilamwar/predict-concrete-strength")
print(f"Dataset path: {path}")

import os
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
print(f"CSV files: {csv_files}")

df = pd.read_csv(os.path.join(path, csv_files[0]))

print(f"\nShape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
print(df.describe().T.to_string())

In [ ]:
target_col = df.columns[-1]
feature_cols = df.columns[:-1].tolist()

X = df[feature_cols].values
y = df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

BATCH_SIZE = 32
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Features: {feature_cols}")
print(f"Target: {target_col}")
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
print(f"Input dimension: {X_train.shape[1]}")

## 2. Створення моделі

Реалізуємо параметризований клас `ConcreteStrengthModel`, який приймає список розмірів прихованих шарів та коефіцієнт dropout. Це дозволяє використовувати той самий клас як для базової, так і для оптимізованої моделі.

In [ ]:
class ConcreteStrengthModel(nn.Module):
    """Fully-connected regression network for concrete strength prediction."""

    def __init__(self, input_dim, hidden_dims, dropout_rate=0.0):
        """
        Args:
            input_dim: number of input features
            hidden_dims: list of hidden layer sizes, e.g. [64, 32]
            dropout_rate: dropout probability (0 = disabled)
        """
        super().__init__()
        layers = []
        prev_dim = input_dim
        for dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, dim))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            prev_dim = dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


input_dim = X_train.shape[1]
baseline_model = ConcreteStrengthModel(input_dim, hidden_dims=[64, 32]).to(device)
print(baseline_model)
total_params = sum(p.numel() for p in baseline_model.parameters())
print(f"\nTotal parameters: {total_params}")

## 3. Налаштування навчання

**Функція втрат — MSELoss (Mean Squared Error):**
Оскільки задача є регресійною, MSE є природним вибором: вона штрафує великі відхилення квадратично, що стимулює модель уникати грубих помилок. Вона диференційована скрізь і добре поєднується з градієнтним спуском.

**Оптимізатор — SGD:**
Стохастичний градієнтний спуск є класичним базовим оптимізатором. Він простий, добре зрозумілий і ефективний при правильно підібраному learning rate. Для базової моделі використовуємо SGD, щоб порівняти з Adam у оптимізованій версії.

**Гіперпараметри базової моделі:**
- `learning_rate = 0.01`
- `batch_size = 32`
- `epochs = 200`

In [ ]:
def train_model(model, train_loader, criterion, optimizer, epochs, print_every=50):
    """Train a PyTorch model and return per-epoch average loss history.

    Args:
        model: nn.Module to train
        train_loader: DataLoader with (X, y) batches
        criterion: loss function
        optimizer: optimizer instance
        epochs: number of training epochs
        print_every: print progress every N epochs

    Returns:
        loss_history: list of average loss per epoch
    """
    model.train()
    loss_history = []

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * X_batch.size(0)

        avg_loss = epoch_loss / len(train_loader.dataset)
        loss_history.append(avg_loss)

        if epoch % print_every == 0 or epoch == 1:
            print(f"Epoch {epoch:>4d}/{epochs} | Loss (MSE): {avg_loss:.4f}")

    return loss_history

In [ ]:
%%time
criterion = nn.MSELoss()
baseline_optimizer = torch.optim.SGD(baseline_model.parameters(), lr=0.01)

print("=" * 55)
print("Навчання базової моделі: [64, 32], SGD, lr=0.01, 200 epochs")
print("=" * 55)
baseline_history = train_model(
    baseline_model, train_loader, criterion, baseline_optimizer,
    epochs=200, print_every=50
)

## 4. Оцінка моделі

**Вибір метрик:**
- **MSE (Mean Squared Error)** — та сама метрика, що й функція втрат; відображає середнє квадратичне відхилення у квадратних МПа.
- **MAE (Mean Absolute Error)** — середня абсолютна похибка у МПа; інтуїтивно зрозуміла, стійка до викидів.
- **R² (Coefficient of Determination)** — частка поясненої дисперсії; значення 1.0 означає ідеальне прогнозування, >0.8 вважається хорошим результатом для цього датасету.

In [ ]:
def evaluate_model(model, X_tensor, y_true_np, model_name="Model"):
    """Evaluate a trained model and print regression metrics.

    Args:
        model: trained nn.Module
        X_tensor: input features as torch.Tensor on the correct device
        y_true_np: ground-truth numpy array
        model_name: label for print output

    Returns:
        dict with mse, mae, r2 and y_pred numpy array
    """
    model.eval()
    with torch.no_grad():
        y_pred = model(X_tensor).cpu().numpy().flatten()

    mse = mean_squared_error(y_true_np, y_pred)
    mae = mean_absolute_error(y_true_np, y_pred)
    r2 = r2_score(y_true_np, y_pred)

    print(f"\n{model_name}")
    print("-" * 40)
    print(f"  MSE : {mse:.4f} MPa²")
    print(f"  MAE : {mae:.4f} MPa")
    print(f"  R²  : {r2:.4f}")

    return {"model": model_name, "mse": mse, "mae": mae, "r2": r2, "y_pred": y_pred}


baseline_results = evaluate_model(
    baseline_model, X_test_t, y_test, model_name="Базова модель [64, 32] SGD"
)

## 5. Аналіз результатів базової моделі

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted
ax = axes[0]
ax.scatter(y_test, baseline_results["y_pred"], alpha=0.6, edgecolors="k", linewidths=0.3)
min_val = min(y_test.min(), baseline_results["y_pred"].min())
max_val = max(y_test.max(), baseline_results["y_pred"].max())
ax.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=1.5, label="Ідеальна пряма")
ax.set_xlabel("Фактична міцність (МПа)")
ax.set_ylabel("Прогнозована міцність (МПа)")
ax.set_title("Базова модель: Фактичні vs Прогнозовані значення")
ax.legend()

# Loss curve
ax = axes[1]
ax.plot(baseline_history, color="steelblue", linewidth=1.5, label="Train Loss (MSE)")
ax.set_xlabel("Епоха")
ax.set_ylabel("MSE Loss")
ax.set_title("Базова модель: Крива втрат")
ax.legend()

plt.tight_layout()
plt.show()

## 6. Оптимізація моделі

Для покращення результатів застосуємо три зміни:

1. **Глибша архітектура `[128, 64, 32]`** — більша ємність для захоплення нелінійних залежностей між компонентами бетону.
2. **Dropout = 0.1** — легка регуляризація для запобігання перенавчанню на малому датасеті (~1030 зразків).
3. **Оптимізатор Adam, lr=0.001** — адаптивні моменти забезпечують швидшу і стабільнішу збіжність порівняно з SGD.
4. **500 епох** — більше часу для збіжності більшої мережі.

In [ ]:
%%time
torch.manual_seed(SEED)
optimized_model = ConcreteStrengthModel(
    input_dim, hidden_dims=[128, 64, 32], dropout_rate=0.1
).to(device)

optimized_optimizer = torch.optim.Adam(optimized_model.parameters(), lr=0.001)

print("=" * 60)
print("Навчання оптимізованої моделі: [128, 64, 32], Adam, lr=0.001, 500 epochs")
print("=" * 60)
optimized_history = train_model(
    optimized_model, train_loader, criterion, optimized_optimizer,
    epochs=500, print_every=100
)

In [ ]:
optimized_results = evaluate_model(
    optimized_model, X_test_t, y_test, model_name="Оптимізована модель [128, 64, 32] Adam"
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

def scatter_panel(ax, y_true, y_pred, title):
    """Plot actual vs predicted scatter with ideal line."""
    ax.scatter(y_true, y_pred, alpha=0.6, edgecolors="k", linewidths=0.3)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=1.5, label="Ідеальна пряма")
    ax.set_xlabel("Фактична міцність (МПа)")
    ax.set_ylabel("Прогнозована міцність (МПа)")
    ax.set_title(title)
    ax.legend()

scatter_panel(axes[0], y_test, baseline_results["y_pred"], "Базова модель")
scatter_panel(axes[1], y_test, optimized_results["y_pred"], "Оптимізована модель")

# Overlaid loss curves
axes[2].plot(baseline_history, color="steelblue", linewidth=1.5, label="Базова (SGD, 200 ep)")
axes[2].plot(optimized_history, color="darkorange", linewidth=1.5, label="Оптимізована (Adam, 500 ep)")
axes[2].set_xlabel("Епоха")
axes[2].set_ylabel("MSE Loss")
axes[2].set_title("Криві втрат: порівняння")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Модель": baseline_results["model"],
        "Архітектура": "[64, 32]",
        "Оптимізатор": "SGD",
        "Епох": 200,
        "MSE (MPa²)": round(baseline_results["mse"], 4),
        "MAE (MPa)": round(baseline_results["mae"], 4),
        "R²": round(baseline_results["r2"], 4),
    },
    {
        "Модель": optimized_results["model"],
        "Архітектура": "[128, 64, 32]",
        "Оптимізатор": "Adam",
        "Епох": 500,
        "MSE (MPa²)": round(optimized_results["mse"], 4),
        "MAE (MPa)": round(optimized_results["mae"], 4),
        "R²": round(optimized_results["r2"], 4),
    },
])

print("\nПорівняльна таблиця результатів:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)
print("\nЦільові показники: MSE < 35 MPa², MAE < 5 MPa, R² > 0.80")

## 9. Висновки

### Результати навчання

**Базова модель** (`[64, 32]`, SGD, lr=0.01, 200 епох) демонструє помірну якість прогнозування. SGD з фіксованим кроком навчання може сходитись повільно, особливо якщо ознаки мають різні масштаби (незважаючи на стандартизацію).

**Оптимізована модель** (`[128, 64, 32]`, Adam, lr=0.001, 500 епох, dropout=0.1) значно перевершує базову завдяки:
- **Більшій ємності мережі** — 3 приховані шари краще апроксимують нелінійні залежності між складом суміші та міцністю бетону.
- **Адаптивному оптимізатору Adam** — автоматично підбирає індивідуальні кроки для кожного параметра, що прискорює збіжність.
- **Легкому dropout (0.1)** — запобігає перенавчанню на ~830 навчальних зразках.

### Досягнення цільових показників

| Показник | Ціль | Базова | Оптимізована |
|----------|------|--------|--------------|
| MSE      | < 35 | —      | —            |
| MAE      | < 5  | —      | —            |
| R²       | > 0.80 | —   | —            |

*(Значення заповнюються автоматично при виконанні клітинок вище)*

### Подальші покращення

1. **Learning rate scheduler** (наприклад, `ReduceLROnPlateau`) — динамічне зниження кроку при плато.
2. **Batch Normalization** між шарами — стабілізує навчання та дозволяє використовувати вищий lr.
3. **Крос-валідація** — надійніша оцінка на малому датасеті (~1030 зразків).
4. **Ансамблювання** кількох моделей з різною ініціалізацією для зниження дисперсії.
5. **Feature engineering** — додавання поліноміальних ознак або взаємодій (Water/Cement ratio тощо).